In [50]:
# Install some packages
%pip install gdown

Note: you may need to restart the kernel to use updated packages.


 Download the file from google drive

In [51]:
# Download the file from google drive
import os
import gdown
import pandas as pd

file_id = '13UKhXw94Dh9XVAcWwxSx1FaKU8_JVY6A'
url = f'https://drive.google.com/uc?id={file_id}'
output_path = 'data.csv'

# Download the file only if it does not exist
if not os.path.exists(output_path):
	try:
		gdown.download(url, output_path, quiet=False)
		print("Data Downloaded")
	except Exception as e:
		print("Could not download file with gdown. Please make sure the file is shared as 'Anyone with the link' and try again.")
		print("Alternatively, download the file manually from the following URL and place it as 'data.csv' in your working directory:")
		print(url)
		print("If you have already downloaded the file, you can skip this step.")
else:
	print("File already exists. Skipping download.")

File already exists. Skipping download.


In [52]:
import pandas as pd

# Load the dataset
file_path = './data.csv'
data = pd.read_csv(file_path, low_memory=False)
print(data.head)

<bound method NDFrame.head of          VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0             1.0  2020-01-01 00:28:15   2020-01-01 00:33:03              1.0   
1             1.0  2020-01-01 00:35:39   2020-01-01 00:43:04              1.0   
2             1.0  2020-01-01 00:47:41   2020-01-01 00:53:52              1.0   
3             1.0  2020-01-01 00:55:23   2020-01-01 01:00:14              1.0   
4             2.0  2020-01-01 00:01:58   2020-01-01 00:04:16              1.0   
...           ...                  ...                   ...              ...   
6405003       NaN  2020-01-31 22:51:00   2020-01-31 23:22:00              NaN   
6405004       NaN  2020-01-31 22:10:00   2020-01-31 23:26:00              NaN   
6405005       NaN  2020-01-31 22:50:07   2020-01-31 23:17:57              NaN   
6405006       NaN  2020-01-31 22:25:53   2020-01-31 22:48:32              NaN   
6405007       NaN  2020-01-31 22:44:00   2020-01-31 23:06:00              NaN  

In [53]:
# Convert datetime columns to proper datetime format if they exist in the dataset
if 'tpep_pickup_datetime' in data.columns and 'tpep_dropoff_datetime' in data.columns:
    data['tpep_pickup_datetime'] = pd.to_datetime(data['tpep_pickup_datetime'])
    data['tpep_dropoff_datetime'] = pd.to_datetime(data['tpep_dropoff_datetime'])
    

QN 1: Extract all trips with trip_distance larger than 50

In [54]:
# QN 1: Extract all trips with trip_distance larger than 50
trips_large_distance = data[data['trip_distance'] > 50]
print(trips_large_distance.head())

       VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
23842       2.0  2020-01-01 01:53:07   2020-01-01 03:54:41              1.0   
39013       2.0  2020-01-01 02:05:07   2020-01-01 03:03:10              1.0   
41620       1.0  2020-01-01 03:05:54   2020-01-01 04:16:26              1.0   
58262       2.0  2020-01-01 05:36:12   2020-01-01 06:40:06              1.0   
63024       2.0  2020-01-01 07:40:30   2020-01-01 08:40:01              1.0   

       trip_distance  RatecodeID store_and_fwd_flag  PULocationID  \
23842          52.30         5.0                  N           262   
39013          51.23         5.0                  N           264   
41620          53.80         5.0                  N           132   
58262          55.23         5.0                  N           132   
63024          54.19         5.0                  N           132   

       DOLocationID  payment_type  fare_amount  extra  mta_tax  tip_amount  \
23842           265           1.

QN 2: Extract all trips where payment_type is missing

In [55]:
# QN 2: Extract all trips where payment_type is missing
trips_missing_payment = data[data['payment_type'].isnull()]
print("Number of trips with missing payment type:", len(trips_missing_payment))
print(trips_missing_payment.head())

Number of trips with missing payment type: 65441
         VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
6339567       NaN  2020-01-01 08:51:00   2020-01-01 09:19:00              NaN   
6339568       NaN  2020-01-01 08:38:43   2020-01-01 08:51:08              NaN   
6339569       NaN  2020-01-01 08:27:00   2020-01-01 08:32:00              NaN   
6339570       NaN  2020-01-01 08:46:00   2020-01-01 08:57:00              NaN   
6339571       NaN  2020-01-01 08:21:00   2020-01-01 08:38:00              NaN   

         trip_distance  RatecodeID store_and_fwd_flag  PULocationID  \
6339567          13.69         NaN                NaN           136   
6339568           3.42         NaN                NaN           121   
6339569           2.20         NaN                NaN           197   
6339570           0.84         NaN                NaN           262   
6339571           7.24         NaN                NaN            45   

         DOLocationID  payment_type  

QN 3: For each (PULocationID, DOLocationID) pair, determine the number of trips

In [56]:
# QN 3: For each (PULocationID, DOLocationID) pair, determine the number of trips
trips_per_location_pair = data.groupby(['PULocationID', 'DOLocationID']).size().reset_index(name='trip_count')
print(trips_per_location_pair.head())

   PULocationID  DOLocationID  trip_count
0             1             1         638
1             1            50           1
2             1            68           1
3             1           138           2
4             1           140           1


QN 4: Save rows with missing VendorID, passenger_count, store_and_fwd_flag, payment_type

In [57]:
# QN 4: Save rows with missing VendorID, passenger_count, store_and_fwd_flag, payment_type
columns_to_check = ['VendorID', 'passenger_count', 'store_and_fwd_flag', 'payment_type']
bad_rows = data[data[columns_to_check].isnull().any(axis=1)]
bad_rows.to_csv('bad_rows.csv', index=False)  # Save bad rows to a new file
data_cleaned = data.drop(bad_rows.index)      # Remove bad rows from the original dataframe

In [58]:
# Printing bad rows
print(f"Number of bad rows: {len(bad_rows)}")
print(bad_rows.head())

Number of bad rows: 65441
         VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
6339567       NaN  2020-01-01 08:51:00   2020-01-01 09:19:00              NaN   
6339568       NaN  2020-01-01 08:38:43   2020-01-01 08:51:08              NaN   
6339569       NaN  2020-01-01 08:27:00   2020-01-01 08:32:00              NaN   
6339570       NaN  2020-01-01 08:46:00   2020-01-01 08:57:00              NaN   
6339571       NaN  2020-01-01 08:21:00   2020-01-01 08:38:00              NaN   

         trip_distance  RatecodeID store_and_fwd_flag  PULocationID  \
6339567          13.69         NaN                NaN           136   
6339568           3.42         NaN                NaN           121   
6339569           2.20         NaN                NaN           197   
6339570           0.84         NaN                NaN           262   
6339571           7.24         NaN                NaN            45   

         DOLocationID  payment_type  fare_amount  extra  mta

In [59]:
# Printing cleaned rows
print(f"Number of rows after cleaning: {len(data_cleaned)}")
print(data_cleaned.head())

Number of rows after cleaning: 6339567
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0       1.0  2020-01-01 00:28:15   2020-01-01 00:33:03              1.0   
1       1.0  2020-01-01 00:35:39   2020-01-01 00:43:04              1.0   
2       1.0  2020-01-01 00:47:41   2020-01-01 00:53:52              1.0   
3       1.0  2020-01-01 00:55:23   2020-01-01 01:00:14              1.0   
4       2.0  2020-01-01 00:01:58   2020-01-01 00:04:16              1.0   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0            1.2         1.0                  N           238           239   
1            1.2         1.0                  N           239           238   
2            0.6         1.0                  N           238           238   
3            0.8         1.0                  N           238           151   
4            0.0         1.0                  N           193           193   

   payment_type  fare_amount  extra

QN 5: Add a duration column storing how long each trip has taken

In [60]:
# QN 5: Add a duration column storing how long each trip has taken
data_cleaned['duration'] = (data_cleaned['tpep_dropoff_datetime'] - data_cleaned['tpep_pickup_datetime']).dt.total_seconds() /60
print(data_cleaned[['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'duration']].head())

  tpep_pickup_datetime tpep_dropoff_datetime  duration
0  2020-01-01 00:28:15   2020-01-01 00:33:03  4.800000
1  2020-01-01 00:35:39   2020-01-01 00:43:04  7.416667
2  2020-01-01 00:47:41   2020-01-01 00:53:52  6.183333
3  2020-01-01 00:55:23   2020-01-01 01:00:14  4.850000
4  2020-01-01 00:01:58   2020-01-01 00:04:16  2.300000


QN 6: For each pickup location, determine how many trips have started there

In [61]:

# QN 6: For each pickup location, determine how many trips have started there
trips_per_pickup = data_cleaned['PULocationID'].value_counts().reset_index(name='trip_count').sort_values(by='PULocationID', ascending=True)
trips_per_pickup.columns = ['PULocationID', 'trip_count']
print(trips_per_pickup)

     PULocationID  trip_count
97              1         753
250             2           3
206             3          70
55              4        9902
225             5          39
..            ...         ...
45            261       34229
30            262       85591
23            263      123997
43            264       43779
74            265        3090

[260 rows x 2 columns]


In [62]:
# QN 6: For each pickup location, determine how many trips have started there
pickup_trip_counts = data_cleaned.groupby('PULocationID').size().reset_index(name='trip_count')
print(pickup_trip_counts.head())

   PULocationID  trip_count
0             1         753
1             2           3
2             3          70
3             4        9902
4             5          39


QN 7: Cluster the pickup time of the day into 30-minute intervals

In [63]:
# QN 7: Cluster the pickup time of the day into 30-minute intervals
data_cleaned['pickup_interval'] = data_cleaned['tpep_pickup_datetime'].dt.floor('30min') # .floor('30min'): Rounds each datetime value down to the nearest 30-minute interval.
print(data_cleaned[['tpep_pickup_datetime', 'pickup_interval']].head())

  tpep_pickup_datetime     pickup_interval
0  2020-01-01 00:28:15 2020-01-01 00:00:00
1  2020-01-01 00:35:39 2020-01-01 00:30:00
2  2020-01-01 00:47:41 2020-01-01 00:30:00
3  2020-01-01 00:55:23 2020-01-01 00:30:00
4  2020-01-01 00:01:58 2020-01-01 00:00:00


QN 8: For each interval, determine the average number of passengers and the average fare amount

In [64]:
# QN 8: For each interval, determine the average number of passengers and the average fare amount
interval_stats = data_cleaned.groupby('pickup_interval').agg({
    'passenger_count': 'mean',
    'fare_amount': 'mean'
}).reset_index()
print(interval_stats.head())

      pickup_interval  passenger_count  fare_amount
0 2003-01-01 00:00:00         1.000000     0.000000
1 2008-12-31 23:00:00         2.000000    18.857143
2 2008-12-31 23:30:00         1.000000    22.666667
3 2009-01-01 00:00:00         1.777778    25.166667
4 2009-01-01 00:30:00         2.000000    24.250000


QN 9: For each payment type and each interval, determine the average fare amount

In [65]:
# QN 9: For each payment type and each interval, determine the average fare amount
fare_by_payment_interval = data_cleaned.groupby(['payment_type', 'pickup_interval'])['fare_amount'].mean().reset_index()
print(fare_by_payment_interval.head())

   payment_type     pickup_interval  fare_amount
0           1.0 2008-12-31 23:00:00        13.50
1           1.0 2008-12-31 23:30:00        52.00
2           1.0 2009-01-01 00:00:00        15.00
3           1.0 2019-12-18 15:00:00         0.01
4           1.0 2019-12-18 15:30:00         2.50


QN 10: For each payment type, determine the interval when the average fare amount is maximum

In [66]:
# QN 10: For each payment type, determine the interval when the average fare amount is maximum
max_fare_interval = fare_by_payment_interval.loc[fare_by_payment_interval.groupby('payment_type')['fare_amount'].idxmax()]
print(max_fare_interval.head())

      payment_type     pickup_interval  fare_amount
1              1.0 2008-12-31 23:30:00      52.0000
1527           2.0 2009-01-01 00:00:00      26.4375
3766           3.0 2020-01-16 03:30:00      45.8000
5591           4.0 2020-01-23 05:00:00      83.8000
6011           5.0 2020-01-21 17:30:00       0.0000


QN 11: For each payment type, determine the interval when the tip-to-fare ratio is maximum

In [67]:
# QN 11: For each payment type, determine the interval when the tip-to-fare ratio is maximum
# Drop rows with NaN in payment_type, pickup_interval, tip_amount, or fare_amount
filtered = data_cleaned.dropna(subset=['payment_type', 'pickup_interval', 'tip_amount', 'fare_amount'])
filtered = filtered[filtered['fare_amount'] != 0]  # Avoid division by zero

filtered['tip_fare_ratio'] = filtered['tip_amount'] / filtered['fare_amount']
max_ratio_interval = filtered.groupby(['payment_type', 'pickup_interval'])['tip_fare_ratio'].mean().reset_index()
max_tip_ratio_interval = max_ratio_interval.loc[max_ratio_interval.groupby('payment_type')['tip_fare_ratio'].idxmax()]
print(max_ratio_interval.head())

   payment_type     pickup_interval  tip_fare_ratio
0           1.0 2008-12-31 23:00:00        0.192847
1           1.0 2008-12-31 23:30:00        0.236154
2           1.0 2009-01-01 00:00:00        0.250667
3           1.0 2019-12-18 15:00:00        0.000000
4           1.0 2019-12-18 15:30:00        0.000000


QN 12: Find the location with the highest average fare amount

In [68]:
# QN 12: Find the location with the highest average fare amount
highest_fare_location = data_cleaned.groupby('PULocationID')['fare_amount'].mean().idxmax()
print("\nLocation with the highest average fare amount:")
print(highest_fare_location)


Location with the highest average fare amount:
204


QN 13: Build a dataframe for each pickup location and its 5 most common destinations

In [69]:
# QN 13: Build a dataframe for each pickup location and its 5 most common destinations
common_destinations = data_cleaned.groupby('PULocationID')['DOLocationID'].value_counts().groupby(level=0).nlargest(5).reset_index(level=0, drop=True).reset_index()
common_destinations.columns = ['PULocationID', 'DOLocationID', 'trip_count']
print(common_destinations.head())

   PULocationID  DOLocationID  trip_count
0             1             1         636
1             1           264         105
2             1           265           4
3             1           138           2
4             1            50           1


QN 14: For each payment type and interval in the common dataframe, determine the average fare amount

In [70]:
# QN 14: For each payment type and interval in the common dataframe, determine the average fare amount
common_data = data_cleaned.merge(common_destinations, on=['PULocationID', 'DOLocationID'], how='inner')
common_fare_stats = common_data.groupby(['payment_type', 'pickup_interval'])['fare_amount'].mean().reset_index()
print(common_fare_stats)

      payment_type     pickup_interval  fare_amount
0              1.0 2008-12-31 23:00:00     6.000000
1              1.0 2019-12-18 15:00:00     0.010000
2              1.0 2019-12-18 15:30:00     2.500000
3              1.0 2019-12-31 14:00:00     6.000000
4              1.0 2019-12-31 16:30:00     7.000000
...            ...                 ...          ...
5891           4.0 2020-01-31 22:00:00     1.187500
5892           4.0 2020-01-31 22:30:00    16.000000
5893           4.0 2020-01-31 23:00:00    -1.833333
5894           4.0 2020-01-31 23:30:00   -16.437500
5895           5.0 2020-01-21 17:30:00     0.000000

[5896 rows x 3 columns]


QN 15: Compute the difference of the average fare amount with those computed in point 9

In [89]:
# QN 15: Compute the difference of the average fare amount with those computed in point 9
# Merge the two DataFrames on payment_type and pickup_interval
common_fare_stats = common_fare_stats.merge(fare_by_payment_interval, on=['payment_type', 'pickup_interval'], suffixes=('_common', '_overall'))
common_fare_stats['fare_diff'] = common_fare_stats['fare_amount_common'] - common_fare_stats['fare_amount_overall']

print(common_fare_stats.head())


   payment_type     pickup_interval  fare_amount_common  fare_amount_overall  \
0           1.0 2008-12-31 23:00:00                6.00                13.50   
1           1.0 2019-12-18 15:00:00                0.01                 0.01   
2           1.0 2019-12-18 15:30:00                2.50                 2.50   
3           1.0 2019-12-31 14:00:00                6.00                12.75   
4           1.0 2019-12-31 16:30:00                7.00                 7.00   

   fare_diff  fare_diff_ratio  fare_amount  
0      -7.50        -0.555556        13.50  
1       0.00         0.000000         0.01  
2       0.00         0.000000         2.50  
3      -6.75        -0.529412        12.75  
4       0.00         0.000000         7.00  


QN 16: Compute the ratio of the differences to the overall fare amounts

In [90]:
# QN 16: Compute the ratio of the differences to the overall fare amounts
common_fare_stats['fare_diff_ratio'] = common_fare_stats['fare_diff'] / common_fare_stats['fare_amount']
print(common_fare_stats.head())  # Shows the first 5 rows


   payment_type     pickup_interval  fare_amount_common  fare_amount_overall  \
0           1.0 2008-12-31 23:00:00                6.00                13.50   
1           1.0 2019-12-18 15:00:00                0.01                 0.01   
2           1.0 2019-12-18 15:30:00                2.50                 2.50   
3           1.0 2019-12-31 14:00:00                6.00                12.75   
4           1.0 2019-12-31 16:30:00                7.00                 7.00   

   fare_diff  fare_diff_ratio  fare_amount  
0      -7.50        -0.555556        13.50  
1       0.00         0.000000         0.01  
2       0.00         0.000000         2.50  
3      -6.75        -0.529412        12.75  
4       0.00         0.000000         7.00  


QN 17: Build chains of trips

In [ ]:
data_cleaned['tpep_pickup_datetime'] = pd.to_datetime(data_cleaned['tpep_pickup_datetime'])
data_cleaned['tpep_dropoff_datetime'] = pd.to_datetime(data_cleaned['tpep_dropoff_datetime'])

# Adding a chain column and identifying chains based on VendorID, locations, and time constraints
data_cleaned = data_cleaned.sort_values(by=['VendorID', 'tpep_pickup_datetime'])
data_cleaned

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,duration,pickup_interval,chain
8483,1.0,2020-01-01 00:00:00,2020-01-01 00:13:03,1.0,2.20,1.0,N,68,170,1.0,...,3.0,0.5,2.85,0.0,0.3,17.15,2.5,13.050000,2020-01-01 00:00:00,8484
3083,1.0,2020-01-01 00:00:03,2020-01-01 00:13:04,2.0,3.00,1.0,N,79,162,1.0,...,3.0,0.5,2.00,0.0,0.3,17.30,2.5,13.016667,2020-01-01 00:00:00,3084
5760,1.0,2020-01-01 00:00:05,2020-01-01 00:26:30,0.0,3.40,1.0,N,264,68,1.0,...,3.0,0.5,4.35,0.0,0.3,26.15,2.5,26.416667,2020-01-01 00:00:00,5761
5050,1.0,2020-01-01 00:00:07,2020-01-01 00:03:26,3.0,0.60,1.0,N,75,75,2.0,...,0.5,0.5,0.00,0.0,0.3,5.80,0.0,3.316667,2020-01-01 00:00:00,5051
2555,1.0,2020-01-01 00:00:25,2020-01-01 00:05:59,1.0,1.70,1.0,N,145,179,2.0,...,0.5,0.5,0.00,0.0,0.3,8.30,0.0,5.566667,2020-01-01 00:00:00,2556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4269480,2.0,2020-07-10 11:34:11,2020-07-10 11:42:41,1.0,1.07,1.0,N,236,262,1.0,...,1.0,0.5,2.26,0.0,0.3,13.56,2.5,8.500000,2020-07-10 11:30:00,4269444
4282277,2.0,2020-07-31 18:50:41,2020-07-31 18:54:12,1.0,0.72,1.0,N,236,43,2.0,...,1.0,0.5,0.00,0.0,0.3,9.30,2.5,3.516667,2020-07-31 18:30:00,4282241
275044,2.0,2021-01-02 00:22:00,2021-01-02 00:36:50,1.0,1.56,1.0,N,142,161,2.0,...,1.0,0.5,0.00,0.0,0.3,14.80,2.5,14.833333,2021-01-02 00:00:00,275040
275045,2.0,2021-01-02 00:44:08,2021-01-02 00:58:56,1.0,2.32,1.0,N,170,148,2.0,...,1.0,0.5,0.00,0.0,0.3,15.80,2.5,14.800000,2021-01-02 00:30:00,275041


In [92]:
# QN 17: Build chains of trips
# Ensure datetime columns are in datetime format
data_cleaned['tpep_pickup_datetime'] = pd.to_datetime(data_cleaned['tpep_pickup_datetime'])
data_cleaned['tpep_dropoff_datetime'] = pd.to_datetime(data_cleaned['tpep_dropoff_datetime'])

# Adding a chain column and identifying chains based on VendorID, locations, and time constraints
data_cleaned = data_cleaned.sort_values(by=['VendorID', 'tpep_pickup_datetime'])
data_cleaned['chain'] = 0
chain_id = 1
for i in range(1, len(data_cleaned)):
    prev_trip = data_cleaned.iloc[i - 1]
    curr_trip = data_cleaned.iloc[i]
    if (
        prev_trip['VendorID'] == curr_trip['VendorID'] and
        prev_trip['DOLocationID'] == curr_trip['PULocationID'] and
        0 < (curr_trip['tpep_pickup_datetime'] - prev_trip['tpep_dropoff_datetime']).total_seconds() <= 120
    ):
        data_cleaned.at[i, 'chain'] = chain_id
        # print("True", data_cleaned.at[i, 'chain'])
    else:
        chain_id += 1
        data_cleaned.at[i, 'chain'] = chain_id
        # print("False", data_cleaned.at[i, 'chain'])
        # print(data_cleaned[['VendorID', 'PULocationID', 'DOLocationID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'chain']])
print(data_cleaned[['VendorID', 'PULocationID', 'DOLocationID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'chain']].head())        


      VendorID  PULocationID  DOLocationID tpep_pickup_datetime  \
8483       1.0            68           170  2020-01-01 00:00:00   
3083       1.0            79           162  2020-01-01 00:00:03   
5760       1.0           264            68  2020-01-01 00:00:05   
5050       1.0            75            75  2020-01-01 00:00:07   
2555       1.0           145           179  2020-01-01 00:00:25   

     tpep_dropoff_datetime  chain  
8483   2020-01-01 00:13:03   8484  
3083   2020-01-01 00:13:04   3084  
5760   2020-01-01 00:26:30   5761  
5050   2020-01-01 00:03:26   5051  
2555   2020-01-01 00:05:59   2556  
